In [1]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from collections import Counter
import json
import re
import hashlib
import unicodedata

import pandas as pd
from IPython.display import display

# Tìm dataset từ thư mục hiện tại hoặc thư mục cha.
candidates = [
    Path.cwd() / "dataset_dan_su_dat_dai",
    Path.cwd().parent / "dataset_dan_su_dat_dai"
]

DATA_DIR = next(
    (
        path.resolve()
        for path in candidates
        if (path / "candidates_raw.jsonl").is_file()
    ),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy candidates_raw.jsonl.\n"
        f"Thư mục làm việc hiện tại: {Path.cwd()}\n"
        "Kiểm tra dataset_dan_su_dat_dai có nằm trong L-GPT không."
    )

RAW_FILE = DATA_DIR / "candidates_raw.jsonl"
OUT_DIR = DATA_DIR / "preprocessing_v01"
OUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
read_errors = []

with RAW_FILE.open("r", encoding="utf-8-sig") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)

            if not isinstance(record, dict):
                raise ValueError("Bản ghi không phải object JSON")

            if not isinstance(record.get("source_record"), dict):
                raise ValueError("Thiếu source_record hợp lệ")

            records.append(record)

        except (json.JSONDecodeError, ValueError) as error:
            read_errors.append({
                "line_number": line_number,
                "error": str(error)
            })

if not records:
    raise RuntimeError("Không đọc được bản ghi hợp lệ nào.")

print("Dữ liệu:", RAW_FILE)
print("Thư mục kết quả:", OUT_DIR)
print("Bản ghi hợp lệ:", len(records))
print("Dòng bị lỗi:", len(read_errors))

Dữ liệu: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\candidates_raw.jsonl
Thư mục kết quả: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01
Bản ghi hợp lệ: 5000
Dòng bị lỗi: 0


In [3]:
def clean_text(value):
    """Chuẩn hóa nhẹ, không viết lại hoặc cắt nội dung."""
    text = unicodedata.normalize("NFC", str(value or ""))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")

    text = "\n".join(
        line.rstrip()
        for line in text.split("\n")
    )

    text = re.sub(r"\n[ \t]*\n(?:[ \t]*\n)+", "\n\n", text)
    return text.strip()


def normalize_search(value):
    """Bỏ dấu chỉ để tìm kiếm; không dùng làm đầu vào LLM."""
    text = unicodedata.normalize("NFD", str(value or "").lower())

    text = "".join(
        char for char in text
        if unicodedata.category(char) != "Mn"
    )

    text = text.replace("đ", "d")
    return re.sub(r"\s+", " ", text).strip()


def hash_text(text):
    normalized = re.sub(r"\s+", " ", text).strip()
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


def detect_document_type(text):
    head = normalize_search(text[:6000])

    patterns = [
        (
            "quyet_dinh_tam_dinh_chi",
            r"quyet dinh\s+tam dinh chi"
        ),
        (
            "quyet_dinh_dinh_chi",
            r"quyet dinh\s+dinh chi"
        ),
        (
            "quyet_dinh_cong_nhan_thoa_thuan",
            r"quyet dinh\s+cong nhan su thoa thuan"
        ),
        (
            "quyet_dinh_cong_nhan_hoa_giai",
            r"quyet dinh\s+cong nhan.{0,100}hoa giai"
        )
    ]

    for label, pattern in patterns:
        if re.search(pattern, head):
            return label

    if re.search(r"ban an\s*(?:so|:|\d)", head):
        return "ban_an"

    return "chua_xac_dinh"


def detect_instance(text):
    head = normalize_search(text[:8000])

    has_appeal = "hoi dong xet xu phuc tham" in head
    has_first = "hoi dong xet xu so tham" in head

    if has_appeal and has_first:
        return "khong_ro"

    if has_appeal:
        return "Appellate"

    if has_first:
        return "First-instance"

    return "chua_xac_dinh"


print("Đã chuẩn bị các hàm xử lý.")

Đã chuẩn bị các hàm xử lý.


In [4]:
inventory_rows = []
processed_records = []

for index, record in enumerate(records, start=1):
    source = record["source_record"]
    screening = record.get("screening") or {}

    raw_text = str(source.get("markdown") or "")
    cleaned_text = clean_text(raw_text)

    document_type = detect_document_type(cleaned_text)
    instance_guess = detect_instance(cleaned_text)
    instance_source = str(source.get("instance_level") or "")

    flags = []

    for field in [
        "doc_name",
        "official_document_id",
        "court",
        "issued_date"
    ]:
        if not source.get(field):
            flags.append(f"missing_{field}")

    if not cleaned_text:
        flags.append("empty_text")
    elif len(cleaned_text) < 3000:
        flags.append("short_text_review")

    if "\ufffd" in cleaned_text:
        flags.append("replacement_character")

    if (
        instance_guess in {"First-instance", "Appellate"}
        and instance_source
        and instance_guess != instance_source
    ):
        flags.append("instance_conflict")

    if instance_guess == "khong_ro":
        flags.append("ambiguous_instance")

    if document_type.startswith("quyet_dinh"):
        flags.append("procedural_decision_review")

    if document_type == "chua_xac_dinh":
        flags.append("unknown_document_type")

    normalized_text = normalize_search(cleaned_text)
    digest = hash_text(cleaned_text)

    inventory_rows.append({
        "record_index": index,
        "doc_name": str(source.get("doc_name") or ""),
        "official_id_source": source.get("official_document_id"),
        "issued_date_source": source.get("issued_date"),
        "court_source": source.get("court"),
        "category_source": source.get("category"),
        "code_source": source.get("code"),
        "instance_source": instance_source,
        "instance_guess": instance_guess,
        "document_type_guess": document_type,
        "candidate_groups": "; ".join(
            screening.get("candidate_groups", [])
        ),
        "chars_raw": len(raw_text),
        "chars_clean": len(cleaned_text),
        "has_content_heading": (
            "noi dung vu an" in normalized_text
        ),
        "has_reasoning_heading": (
            "nhan dinh cua toa an" in normalized_text
        ),
        "web_url": source.get("web_url"),
        "pdf_url": source.get("pdf_url"),
        "text_sha256": digest,
        "quality_flags": "; ".join(flags),

        # Các cột này để người rà soát điền sau.
        "manual_eligible": "",
        "manual_document_type": "",
        "manual_instance": "",
        "case_group_id": "",
        "review_notes": ""
    })

    processed_records.append({
        **record,
        "preprocessing": {
            "version": "v01",
            "text_clean": cleaned_text,
            "text_sha256": digest,
            "document_type_guess": document_type,
            "instance_guess": instance_guess,
            "quality_flags": flags,
            "review_status": "not_reviewed"
        }
    })

df = pd.DataFrame(inventory_rows)

print("Đã xử lý:", len(df), "bản ghi")
display(
    df[
        [
            "doc_name",
            "official_id_source",
            "instance_source",
            "instance_guess",
            "document_type_guess",
            "quality_flags"
        ]
    ].head(10)
)

Đã xử lý: 5000 bản ghi


,doc_name,official_id_source,instance_source,instance_guess,document_type_guess,quality_flags
0,304,117/2017/DS-PT,Appellate,Appellate,ban_an,
1,672,1/2017/DS,,First-instance,ban_an,
2,880,11/2017/DSPT,Appellate,Appellate,ban_an,
3,1744,10/2017/QĐST,First-instance,chua_xac_dinh,quyet_dinh_dinh_chi,procedural_decision_review
4,2256,9/2017/DS-PT,Appellate,Appellate,ban_an,
5,3104,107/2017/QĐST-DS,First-instance,chua_xac_dinh,quyet_dinh_dinh_chi,short_text_review; procedural_decision_review
6,3680,20/2017/QĐST-DS,First-instance,chua_xac_dinh,quyet_dinh_dinh_chi,short_text_review; procedural_decision_review
7,4560,25/2017/DSPT,Appellate,Appellate,ban_an,
8,5216,123/2017/DS-PT,Appellate,Appellate,ban_an,
9,5248,46/2017/DSST,First-instance,Appellate,quyet_dinh_cong_nhan_thoa_thuan,instance_conflict; procedural_decision_review


In [5]:
# Trùng mã nguồn.
df["duplicate_doc_name"] = (
    df["doc_name"].ne("")
    & df.duplicated("doc_name", keep=False)
)

# Trùng toàn văn sau chuẩn hóa khoảng trắng.
df["duplicate_text"] = (
    df["chars_clean"].gt(0)
    & df.duplicated("text_sha256", keep=False)
)

# Nghi trùng theo số bản án + ngày + tòa.
metadata_columns = [
    "official_id_source",
    "issued_date_source",
    "court_source"
]

complete_metadata = pd.Series(True, index=df.index)

for column in metadata_columns:
    complete_metadata &= (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

df["metadata_key"] = (
    df["official_id_source"].fillna("").map(normalize_search)
    + "|"
    + df["issued_date_source"].fillna("").astype(str)
    + "|"
    + df["court_source"].fillna("").map(normalize_search)
)

df["suspected_metadata_duplicate"] = (
    complete_metadata
    & df.duplicated("metadata_key", keep=False)
)

duplicate_mask = (
    df["duplicate_doc_name"]
    | df["duplicate_text"]
    | df["suspected_metadata_duplicate"]
)

df["needs_review"] = (
    df["quality_flags"].ne("")
    | duplicate_mask
)


def assign_review_route(row):
    if row["chars_clean"] == 0:
        return "empty_text"

    if row["document_type_guess"].startswith("quyet_dinh"):
        return "procedural_decision_review"

    if row["needs_review"]:
        return "quality_or_metadata_review"

    if row["document_type_guess"] == "ban_an":
        return "judgment_candidate_review"

    return "other_review"


df["review_route"] = df.apply(assign_review_route, axis=1)

print("PHÂN LOẠI VĂN BẢN SƠ BỘ")
display(
    df["document_type_guess"]
    .value_counts()
    .rename_axis("loai_van_ban")
    .reset_index(name="so_luong")
)

print("CẤP XÉT XỬ THEO NGUỒN")
display(
    df["instance_source"]
    .replace("", "missing")
    .value_counts()
    .rename_axis("cap_xet_xu")
    .reset_index(name="so_luong")
)

print("PHÂN LUỒNG RÀ SOÁT")
display(
    df["review_route"]
    .value_counts()
    .rename_axis("nhom_ra_soat")
    .reset_index(name="so_luong")
)

print("Số bản ghi nghi trùng:", int(duplicate_mask.sum()))
print("Số bản ghi có cờ cần rà soát:", int(df["needs_review"].sum()))

PHÂN LOẠI VĂN BẢN SƠ BỘ


,loai_van_ban,so_luong
0,ban_an,3187
1,quyet_dinh_dinh_chi,1324
2,quyet_dinh_cong_nhan_thoa_thuan,276
3,chua_xac_dinh,172
4,quyet_dinh_tam_dinh_chi,27
5,quyet_dinh_cong_nhan_hoa_giai,14


CẤP XÉT XỬ THEO NGUỒN


,cap_xet_xu,so_luong
0,First-instance,2840
1,Appellate,2048
2,missing,58
3,Cassation,51
4,Retrial,3


PHÂN LUỒNG RÀ SOÁT


,nhom_ra_soat,so_luong
0,judgment_candidate_review,2944
1,procedural_decision_review,1641
2,quality_or_metadata_review,415


Số bản ghi nghi trùng: 20
Số bản ghi có cờ cần rà soát: 2056


In [6]:
from datetime import datetime, timezone
import shutil

RUN_DIR = OUT_DIR / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=False)


def save_csv(table, filename):
    table.to_csv(
        RUN_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )


# Danh mục và danh sách rà soát.
save_csv(df, "inventory_all.csv")
save_csv(df[df["needs_review"]], "review_required.csv")

save_csv(
    df[duplicate_mask].sort_values("metadata_key"),
    "duplicate_candidates.csv"
)

save_csv(
    df[df["review_route"].eq("judgment_candidate_review")],
    "judgment_candidates_review.csv"
)

save_csv(
    df[df["review_route"].eq("procedural_decision_review")],
    "procedural_decisions_review.csv"
)

save_csv(
    pd.DataFrame(
        read_errors,
        columns=["line_number", "error"]
    ),
    "read_errors.csv"
)

# Bảng phân bố.
for column in [
    "category_source",
    "instance_source",
    "document_type_guess",
    "review_route"
]:
    distribution = (
        df[column]
        .fillna("missing")
        .replace("", "missing")
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )

    save_csv(distribution, f"distribution_{column}.csv")

# Phân bố nhóm từ khóa: một bản có thể thuộc nhiều nhóm.
group_counts = Counter(
    group
    for record in records
    for group in (
        (record.get("screening") or {})
        .get("candidate_groups", [])
    )
)

save_csv(
    pd.DataFrame(
        group_counts.most_common(),
        columns=["candidate_group", "count"]
    ),
    "distribution_candidate_groups.csv"
)

# Đồng bộ cờ kiểm kê vào JSONL.
inventory_data = df.to_dict("records")

processed_file = RUN_DIR / "documents_preprocessed.jsonl"

with processed_file.open("w", encoding="utf-8") as file:
    for record, row in zip(processed_records, inventory_data):
        record["preprocessing"].update({
            "duplicate_doc_name": row["duplicate_doc_name"],
            "duplicate_text": row["duplicate_text"],
            "suspected_metadata_duplicate": row[
                "suspected_metadata_duplicate"
            ],
            "review_route": row["review_route"]
        })

        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_file": str(RAW_FILE),
    "valid_records": len(df),
    "invalid_lines": len(read_errors),
    "records_needing_review": int(df["needs_review"].sum()),
    "records_flagged_duplicate": int(duplicate_mask.sum()),
    "review_routes": {
        str(key): int(value)
        for key, value in df["review_route"].value_counts().items()
    },
    "status": "automatic_inventory_not_manually_verified"
}

(RUN_DIR / "inventory_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(RUN_DIR),
    "zip",
    root_dir=RUN_DIR
)

print("ĐÃ HOÀN THÀNH")
print("Thư mục kết quả:", RUN_DIR)
print("File ZIP:", zip_path)
print("\nTóm tắt:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

ĐÃ HOÀN THÀNH
Thư mục kết quả: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613
File ZIP: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613.zip

Tóm tắt:
{
  "created_at_utc": "2026-11-05T08:26:18.490079+00:00",
  "input_file": "C:\\Users\\LENOVO\\OneDrive\\Desktop\\L-GPT\\dataset_dan_su_dat_dai\\candidates_raw.jsonl",
  "valid_records": 5000,
  "invalid_lines": 0,
  "records_needing_review": 2056,
  "records_flagged_duplicate": 20,
  "review_routes": {
    "judgment_candidate_review": 2944,
    "procedural_decision_review": 1641,
    "quality_or_metadata_review": 415
  },
  "status": "automatic_inventory_not_manually_verified"
}


In [7]:
# Ô 7 — Tạo lô 50 bản án để rà soát
import random

BATCH_SIZE = 50
SEED = 42

required = ["df", "processed_records", "RUN_DIR"]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Thiếu biến {missing}. Chạy lại các ô 2–6 trước."
    )

# Lô phát triển quy tắc trích xuất, không phải tập test.
pool = df[
    df["review_route"].eq("judgment_candidate_review")
].copy()

if pool.empty:
    raise RuntimeError("Không có ứng viên trong danh sách ưu tiên.")

# Xáo trộn có seed, sau đó lấy luân phiên giữa các nhóm từ khóa.
# Một bản có thể thuộc nhiều nhóm nhưng chỉ được lấy một lần.
group_names = sorted({
    group
    for value in pool["candidate_groups"]
    for group in str(value).split("; ")
    if group
})

rng = random.Random(SEED)
buckets = {}

for group in group_names:
    indices = pool.loc[
        pool["candidate_groups"].map(
            lambda value: group in str(value).split("; ")
        ),
        "record_index"
    ].tolist()

    rng.shuffle(indices)
    buckets[group] = indices

selected_indices = []
selected_set = set()

while len(selected_indices) < min(BATCH_SIZE, len(pool)):
    added_this_round = False

    for group in group_names:
        bucket = buckets[group]

        while bucket and bucket[-1] in selected_set:
            bucket.pop()

        if bucket:
            index = bucket.pop()
            selected_indices.append(index)
            selected_set.add(index)
            added_this_round = True

        if len(selected_indices) >= BATCH_SIZE:
            break

    if not added_this_round:
        break

# Bổ sung nếu các nhóm từ khóa chưa cung cấp đủ mẫu.
remaining = [
    index
    for index in pool["record_index"].tolist()
    if index not in selected_set
]
rng.shuffle(remaining)

selected_indices.extend(
    remaining[:max(0, BATCH_SIZE - len(selected_indices))]
)

lookup = df.set_index("record_index")
batch = lookup.loc[selected_indices].reset_index()

BATCH_DIR = RUN_DIR / "review_batch_50"
TEXT_DIR = BATCH_DIR / "fulltext"

TEXT_DIR.mkdir(parents=True, exist_ok=False)

review_rows = []

for order, row in enumerate(batch.to_dict("records"), start=1):
    # record_index được tạo từ 1 trong ô xử lý.
    record = processed_records[int(row["record_index"]) - 1]

    source = record["source_record"]
    text = record["preprocessing"]["text_clean"]

    safe_id = re.sub(
        r"[^a-zA-Z0-9_-]",
        "_",
        str(row["doc_name"])
    )

    filename = f"{order:02d}_{safe_id}.md"

    header = (
        f"# Mẫu rà soát {order:02d}\n\n"
        f"- doc_name: {row['doc_name']}\n"
        f"- Số hiệu nguồn: {row['official_id_source']}\n"
        f"- Ngày nguồn: {row['issued_date_source']}\n"
        f"- Tòa nguồn: {row['court_source']}\n"
        f"- Cấp xét xử nguồn: {row['instance_source']}\n"
        f"- Nhóm từ khóa: {row['candidate_groups']}\n"
        f"- Trang nguồn: {source.get('web_url')}\n"
        f"- PDF nguồn: {source.get('pdf_url')}\n\n"
        "---\n\n"
    )

    (TEXT_DIR / filename).write_text(
        header + text,
        encoding="utf-8"
    )

    review_rows.append({
        "review_order": order,
        "doc_name": row["doc_name"],
        "fulltext_file": filename,
        "official_id_source": row["official_id_source"],
        "court_source": row["court_source"],
        "issued_date_source": row["issued_date_source"],
        "instance_source": row["instance_source"],
        "candidate_groups": row["candidate_groups"],
        "web_url": row["web_url"],
        "pdf_url": row["pdf_url"],

        # Người đọc điền các cột sau.
        "actual_official_id": "",
        "actual_court": "",
        "actual_date": "",
        "actual_instance": "",
        "actual_dispute_type": "",
        "eligible": "",

        "request_quote": "",
        "first_instance_outcome_quote": "",
        "outcome_label": "",

        "pre_first_instance_facts_available": "",
        "case_group_id": "",
        "review_notes": ""
    })

review_df = pd.DataFrame(review_rows)

review_df.to_csv(
    BATCH_DIR / "review_50.csv",
    index=False,
    encoding="utf-8-sig"
)

(BATCH_DIR / "batch_info.json").write_text(
    json.dumps({
        "seed": SEED,
        "size": len(review_df),
        "purpose": "development_of_annotation_rules_not_test",
        "selection": "round_robin_keyword_groups_from_priority_pool",
        "doc_names": review_df["doc_name"].tolist()
    }, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(BATCH_DIR),
    "zip",
    root_dir=BATCH_DIR
)

print("Số bản trong lô:", len(review_df))
print("Bảng rà soát:", BATCH_DIR / "review_50.csv")
print("ZIP:", zip_path)

display(
    review_df[
        [
            "review_order",
            "doc_name",
            "instance_source",
            "candidate_groups"
        ]
    ].head(10)
)

Số bản trong lô: 50
Bảng rà soát: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613\review_batch_50\review_50.csv
ZIP: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613\review_batch_50.zip


,review_order,doc_name,instance_source,candidate_groups
0,1,124369,First-instance,chuyen_nhuong_qsdd
1,2,328545,Appellate,ranh_gioi_lan_chiem; dat_dai_khac
2,3,995536,First-instance,doi_nha_dat_o_nho; dat_dai_khac
3,4,635360,First-instance,loi_di_bds_lien_ke; dat_dai_khac
4,5,28849,First-instance,ranh_gioi_lan_chiem; dat_dai_khac
5,6,495568,First-instance,chuyen_nhuong_qsdd
6,7,1801248,First-instance,dat_dai_khac
7,8,50800,First-instance,chuyen_nhuong_qsdd; doi_nha_dat_o_nho; dat_dai...
8,9,1276080,Appellate,loi_di_bds_lien_ke; chuyen_nhuong_qsdd
9,10,672833,Appellate,chuyen_nhuong_qsdd; ranh_gioi_lan_chiem; dat_d...
